In [2]:
! python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1.3 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [6]:
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json

In [5]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
nlp = spacy.load("en_core_web_sm")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11625.45it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
canonicalSymptoms = json.load(open("templates/canonicalSymptoms.json", "r"))

In [8]:
symptomEmbeddings = embedder.encode(canonicalSymptoms)

In [9]:
def extractCandidateSymptoms(text):
    doc = nlp(text)
    phrases = []

    for chunk in doc.noun_chunks:

        cleanCheck = " ".join([t.text for t in chunk if not t.is_stop])
        if cleanCheck:
            phrases.append(cleanCheck)

    for token in doc:
        if token.pos_ in ["ADJ", "VERB"] and not token.is_stop:
            phrases.append(token.lemma_)

    phrases.append(text)

    return list(set(phrases))

In [10]:
def extractSymptoms(patientText, threshold=0.5):
    candidatePhrases = extractCandidateSymptoms(patientText)
    if not candidatePhrases:
        return []
    
    phraseEmbeddings = embedder.encode(candidatePhrases)
    similarityMatrix = cosine_similarity(phraseEmbeddings, symptomEmbeddings)

    extracted = set()

    for i, phrase in enumerate(candidatePhrases):
        bestMatchIdx = np.argmax(similarityMatrix[i])
        bestScore = similarityMatrix[i][bestMatchIdx]

        if bestScore >= threshold:
            extracted.add(canonicalSymptoms[bestMatchIdx])

    return list(extracted)

In [12]:
extractSymptoms("I have been experiencing ear fullness and a buzzing sound in my ear.")

['ear pain', 'buzzing', 'ear fullness']

In [13]:
extractSymptoms("my ear feels heavy and I hear buzzing sometimes")

['ear pain', 'buzzing']